# Lesson 03: Part 2 — Prompt Patterns in Python Code

### Goal
In Lesson 02, we studied prompt patterns. Now we run them on the **exact same customer ticket** to see how the model's output changes:
1. **Zero-Shot:** Instructions only.
2. **One-Shot:** Instructions + 1 example.
3. **Few-Shot:** Instructions + 2 examples.
4. **XML Structure:** Separating rules, examples, and input with XML tags.

---
## Setup: Connect to Gemini API

In [1]:
!pip install --quiet -U google-genai

from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print("Connected to Gemini API successfully!")

Connected to Gemini API successfully!


---
## The Customer Ticket
Here is the customer message we will analyze in all four steps:

In [2]:
ticket = """Hello, I ordered 2 SSD drives (Order #48192). Only 1 arrived today, and the box was opened. Please send the missing drive or refund $90 immediately."""
print("Ticket:", ticket)

Ticket: Hello, I ordered 2 SSD drives (Order #48192). Only 1 arrived today, and the box was opened. Please send the missing drive or refund $90 immediately.


---
## Pattern 1: Zero-Shot (No Examples)
We only ask the model to extract the fields. No examples are provided.

In [3]:
prompt_zero = f"""Extract order_id, missing_items_count, and refund_amount_usd into JSON from this message:
{ticket}"""

resp_zero = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt_zero
)
print(resp_zero.text.strip())

{
  "order_id": "48192",
  "missing_items_count": 1,
  "refund_amount_usd": 90.0
}


---
## Pattern 2: One-Shot (1 Example)
We provide exactly one demonstration example. Notice how the model copies the exact format of the example.

In [4]:
prompt_one = f"""Extract order_id, missing_items_count, and refund_amount_usd into JSON.

Example:
Message: "Order #111 had 3 cables, but only 2 arrived. Need $15 refund."
JSON: {{"order_id": "111", "missing_items_count": 1, "refund_amount_usd": 15.0}}

Message:
{ticket}"""

resp_one = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt_one
)
print(resp_one.text.strip())

{
  "order_id": "48192",
  "missing_items_count": 1,
  "refund_amount_usd": 90.0
}


---
## Pattern 3: Few-Shot (Multiple Examples)
We provide 2 examples, including an edge case with no refund.

In [5]:
prompt_few = f"""Extract order_id, missing_items_count, and refund_amount_usd into JSON.

Example 1:
Message: "Order #111 had 3 cables, but only 2 arrived. Need $15 refund."
JSON: {{"order_id": "111", "missing_items_count": 1, "refund_amount_usd": 15.0}}

Example 2:
Message: "Order #222 is delayed. Nothing is missing and no refund needed."
JSON: {{"order_id": "222", "missing_items_count": 0, "refund_amount_usd": null}}

Message:
{ticket}"""

resp_few = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt_few
)
print(resp_few.text.strip())

{
  "order_id": "48192",
  "missing_items_count": 1,
  "refund_amount_usd": 90.0
}


---
## Pattern 4: XML Structure (Production Standard)
We wrap instructions, examples, and user input inside clean XML tags (`<rules>`, `<examples>`, `<ticket>`).

This makes it impossible for user text to be confused with instructions.

In [6]:
prompt_xml = f"""<rules>
Extract order_id, missing_items_count, refund_amount_usd into valid JSON.
</rules>

<examples>
<example>
<ticket>Order #111 had 3 cables, 1 missing. Refund $15.</ticket>
<json>{{"order_id": "111", "missing_items_count": 1, "refund_amount_usd": 15.0}}</json>
</example>
</examples>

<ticket>
{ticket}
</ticket>"""

resp_xml = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=prompt_xml
)
print(resp_xml.text.strip())

{
  "order_id": "48192",
  "missing_items_count": 1,
  "refund_amount_usd": 90.0
}


---
## Step 5: Try It Yourself (Playground)
Change `my_ticket` below to test your own customer message.

In [7]:
# Try your own customer message:
my_ticket = "Hi, order #77332 arrived broken. All 4 glasses shattered. Refund $60 please."

my_prompt = f"""<rules>
Extract order_id, missing_items_count, refund_amount_usd into valid JSON.
</rules>
<ticket>
{my_ticket}
</ticket>"""

my_resp = client.models.generate_content(
    model='gemini-3.5-flash-lite',
    contents=my_prompt
)
print(my_resp.text.strip())

{
  "order_id": "77332",
  "missing_items_count": 4,
  "refund_amount_usd": 60.0
}
